In [2]:
import pandas as pd
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score ,  precision_recall_curve
import numpy as np




In [3]:
X_train = pd.read_csv('X_train.csv')
y_train = pd.read_csv('y_train.csv').squeeze()
X_test  = pd.read_csv('X_test.csv')
y_test  = pd.read_csv('y_test.csv').squeeze()

In [4]:

X_train_res, y_train_res = SMOTE(random_state=42).fit_resample(X_train, y_train)

In [5]:
models = {
    'LogisticRegression': LogisticRegression(max_iter=1000),
    'RandomForest':       RandomForestClassifier(n_jobs=-1),
    'XGBoost':            XGBClassifier(eval_metric='aucpr', n_jobs=-1),
    'LightGBM':           LGBMClassifier(is_unbalance=True,n_jobs=-1),
}


In [6]:

results = {}
for name, model in models.items():
    
    model.fit(X_train_res, y_train_res)
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    results[name] = {
        'AUC-ROC': roc_auc_score(y_test, y_proba),
        'AUC-PR':  average_precision_score(y_test, y_proba),
        'Report':  classification_report(y_test, y_pred)
    }
    print(f"\n── {name} ──")
    print(f"AUC-ROC: {results[name]['AUC-ROC']:.4f} | AUC-PR: {results[name]['AUC-PR']:.4f}")
    print(results[name]['Report'])

/home/m0-obe/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



── LogisticRegression ──
AUC-ROC: 0.8961 | AUC-PR: 0.1231
              precision    recall  f1-score   support

           0       1.00      0.97      0.98    553574
           1       0.08      0.73      0.14      2145

    accuracy                           0.97    555719
   macro avg       0.54      0.85      0.56    555719
weighted avg       1.00      0.97      0.98    555719


── RandomForest ──
AUC-ROC: 0.9832 | AUC-PR: 0.8106
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    553574
           1       0.68      0.81      0.74      2145

    accuracy                           1.00    555719
   macro avg       0.84      0.90      0.87    555719
weighted avg       1.00      1.00      1.00    555719


── XGBoost ──
AUC-ROC: 0.9964 | AUC-PR: 0.8575
              precision    recall  f1-score   support

           0       1.00      0.99      1.00    553574
           1       0.35      0.91      0.50      2145

    accuracy          

In [7]:

model = RandomForestClassifier(n_jobs=-1,class_weight='balanced')

model.fit(X_train, y_train)
y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

result = {
        'AUC-ROC': roc_auc_score(y_test, y_proba),
        'AUC-PR':  average_precision_score(y_test, y_proba),
        'Report':  classification_report(y_test, y_pred)
    }



In [8]:
print(f"\n── RF-unbalanced ──")
print(f"AUC-ROC: {result['AUC-ROC']:.4f} | AUC-PR: {result['AUC-PR']:.4f}")
print(result['Report'])


── RF-unbalanced ──
AUC-ROC: 0.9742 | AUC-PR: 0.8548
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    553574
           1       0.91      0.73      0.81      2145

    accuracy                           1.00    555719
   macro avg       0.95      0.86      0.90    555719
weighted avg       1.00      1.00      1.00    555719



In [9]:
ratio = y_train.value_counts()[0] / y_train.value_counts()[1]  # ~258

models = {
    "xgb" : XGBClassifier(scale_pos_weight=ratio, n_jobs=-1),
    "lgbm":LGBMClassifier(scale_pos_weight=ratio, n_jobs=-1)
}

In [10]:
results = {}
for name, model in models.items():
    
    model.fit(X_train, y_train)
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    results[name] = {
        'AUC-ROC': roc_auc_score(y_test, y_proba),
        'AUC-PR':  average_precision_score(y_test, y_proba),
        'Report':  classification_report(y_test, y_pred)
    }
    print(f"\n── {name} ──")
    print(f"AUC-ROC: {results[name]['AUC-ROC']:.4f} | AUC-PR: {results[name]['AUC-PR']:.4f}")
    print(results[name]['Report'])


── xgb ──
AUC-ROC: 0.9976 | AUC-PR: 0.8517
              precision    recall  f1-score   support

           0       1.00      0.99      0.99    553574
           1       0.26      0.94      0.41      2145

    accuracy                           0.99    555719
   macro avg       0.63      0.97      0.70    555719
weighted avg       1.00      0.99      0.99    555719

[LightGBM] [Info] Number of positive: 7506, number of negative: 1289169
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.017351 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 407
[LightGBM] [Info] Number of data points in the train set: 1296675, number of used features: 19
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.005789 -> initscore=-5.146050
[LightGBM] [Info] Start training from score -5.146050

── lgbm ──
AUC-ROC: 0.9579 | AUC-PR: 0.1087
              precision 

In [11]:


def evaluate_at_best_threshold(model, X_test, y_test):
    y_proba = model.predict_proba(X_test)[:, 1]
    precision, recall, thresholds = precision_recall_curve(y_test, y_proba)
    f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)
    best_idx = f1_scores.argmax()
    best_threshold = thresholds[best_idx]
    y_pred = (y_proba >= best_threshold).astype(int)
    print(f"Best threshold: {best_threshold:.4f}")
    print(classification_report(y_test, y_pred))
    print(f"AUC-PR: {average_precision_score(y_test, y_proba):.4f}")

In [12]:
rf_model   = RandomForestClassifier(n_jobs=-1, class_weight='balanced').fit(X_train, y_train)
xgb_model  = XGBClassifier(eval_metric='aucpr', n_jobs=-1).fit(X_train_res, y_train_res)
lgbm_model = LGBMClassifier(is_unbalance=True, n_jobs=-1).fit(X_train_res, y_train_res)

[LightGBM] [Info] Number of positive: 1289169, number of negative: 1289169
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.038502 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 409
[LightGBM] [Info] Number of data points in the train set: 2578338, number of used features: 19
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000


In [13]:
models = {
    'RF':       rf_model,
    'XGBoost':  xgb_model,
    'LightGBM': lgbm_model
}

for name, model in models.items():
    print(f"\n── {name} ──")
    evaluate_at_best_threshold(model, X_test, y_test)


── RF ──
Best threshold: 0.4500
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    553574
           1       0.90      0.76      0.82      2145

    accuracy                           1.00    555719
   macro avg       0.95      0.88      0.91    555719
weighted avg       1.00      1.00      1.00    555719

AUC-PR: 0.8550

── XGBoost ──
Best threshold: 0.9815
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    553574
           1       0.89      0.74      0.81      2145

    accuracy                           1.00    555719
   macro avg       0.95      0.87      0.90    555719
weighted avg       1.00      1.00      1.00    555719

AUC-PR: 0.8575

── LightGBM ──
Best threshold: 0.9628
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    553574
           1       0.83      0.76      0.79      2145

    accuracy                           1.0

In [14]:
from sklearn.model_selection import RandomizedSearchCV
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

# ── Random Forest ──────────────────────────────────────────────────────────────
rf_params = {
    'n_estimators':      [100, 200, 300, 500],
    'max_depth':         [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf':  [1, 2, 4],
    'max_features':      ['sqrt', 'log2'],
}

rf_search = RandomizedSearchCV(
    estimator  = RandomForestClassifier(n_jobs=-1, class_weight='balanced'),
    param_distributions = rf_params,
    n_iter     = 20,          # try 20 random combinations
    scoring    = 'average_precision',  # optimize for AUC-PR
    cv         = 3,           # 3-fold cross validation
    n_jobs     = -1,
    random_state = 42,
    verbose    = 2
)
rf_search.fit(X_train, y_train)  # no SMOTE needed for RF
print("Best RF params:", rf_search.best_params_)
print("Best RF AUC-PR:", rf_search.best_score_)




Fitting 3 folds for each of 20 candidates, totalling 60 fits
[CV] END max_depth=10, max_features=log2, min_samples_leaf=1, min_samples_split=10, n_estimators=200; total time= 6.0min
[CV] END max_depth=10, max_features=log2, min_samples_leaf=1, min_samples_split=10, n_estimators=200; total time= 6.0min
[CV] END max_depth=10, max_features=log2, min_samples_leaf=1, min_samples_split=10, n_estimators=200; total time= 6.0min
[CV] END max_depth=30, max_features=sqrt, min_samples_leaf=2, min_samples_split=2, n_estimators=200; total time= 7.3min
[CV] END max_depth=None, max_features=log2, min_samples_leaf=1, min_samples_split=5, n_estimators=100; total time= 3.5min
[CV] END max_depth=None, max_features=log2, min_samples_leaf=1, min_samples_split=5, n_estimators=100; total time= 3.5min
[CV] END max_depth=30, max_features=sqrt, min_samples_leaf=2, min_samples_split=2, n_estimators=200; total time= 6.7min
[CV] END max_depth=None, max_features=log2, min_samples_leaf=1, min_samples_split=5, n_estim

/home/m0-obe/miniconda3/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[CV] END max_depth=10, max_features=log2, min_samples_leaf=1, min_samples_split=5, n_estimators=300; total time= 8.7min
[CV] END max_depth=30, max_features=log2, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time= 7.6min
[CV] END max_depth=10, max_features=log2, min_samples_leaf=1, min_samples_split=5, n_estimators=300; total time= 8.7min
[CV] END max_depth=10, max_features=log2, min_samples_leaf=1, min_samples_split=5, n_estimators=300; total time= 8.6min
[CV] END max_depth=30, max_features=log2, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time= 8.4min
[CV] END max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=10, n_estimators=200; total time= 6.8min
[CV] END max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=10, n_estimators=200; total time= 6.9min
[CV] END max_depth=30, max_features=log2, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time= 8.1min
[CV] END max_depth=10, max_features=sq

In [15]:
# ── XGBoost ────────────────────────────────────────────────────────────────────
xgb_pipe = Pipeline([
    ('smote', SMOTE(random_state=42)),
    ('model', XGBClassifier(eval_metric='aucpr', n_jobs=-1))
])

xgb_params = {
    'model__n_estimators':  [100, 300, 500],
    'model__max_depth':     [3, 5, 7],
    'model__learning_rate': [0.01, 0.05, 0.1],
    'model__subsample':     [0.6, 0.8, 1.0],
    'model__colsample_bytree': [0.6, 0.8, 1.0],
    'model__scale_pos_weight': [1, ratio],  # let it choose between SMOTE-balanced vs weighted
}

xgb_search = RandomizedSearchCV(
    estimator  = xgb_pipe,
    param_distributions = xgb_params,
    n_iter     = 20,
    scoring    = 'average_precision',
    cv         = 3,
    n_jobs     = -1,
    random_state = 42,
    verbose    = 2
)
xgb_search.fit(X_train, y_train)  # original X_train, SMOTE is inside the pipeline
print("Best XGB params:", xgb_search.best_params_)
print("Best XGB AUC-PR:", xgb_search.best_score_)

Fitting 3 folds for each of 20 candidates, totalling 60 fits
[CV] END model__colsample_bytree=1.0, model__learning_rate=0.05, model__max_depth=7, model__n_estimators=300, model__scale_pos_weight=1, model__subsample=0.6; total time= 3.2min
[CV] END model__colsample_bytree=1.0, model__learning_rate=0.05, model__max_depth=7, model__n_estimators=300, model__scale_pos_weight=1, model__subsample=0.6; total time= 3.2min
[CV] END model__colsample_bytree=0.8, model__learning_rate=0.05, model__max_depth=7, model__n_estimators=300, model__scale_pos_weight=171.75179856115108, model__subsample=0.8; total time= 3.3min
[CV] END model__colsample_bytree=1.0, model__learning_rate=0.05, model__max_depth=7, model__n_estimators=300, model__scale_pos_weight=1, model__subsample=0.6; total time= 3.3min


KeyboardInterrupt: 